# 🔥 QueryForge-AI
## Fine-tuning Mistral 7B v0.3 for Natural Language → SQL Generation

**Dataset:** `b-mc2/sql-create-context` (SQLCoder-style Alpaca format)

**Framework:** Unsloth + QLoRA + TRL

**Model:** `unsloth/mistral-7b-v0.3-bnb-4bit`

---
### What this notebook does:
- ✅ Loads Mistral 7B in 4-bit (fits on free T4 GPU)
- ✅ Fine-tunes on SQL question → SQL query pairs
- ✅ Uses schema-aware prompting
- ✅ Saves model as GGUF + pushes to HuggingFace Hub

> **Runtime:** Make sure you have GPU enabled → Runtime > Change runtime type > T4 GPU

## 📦 Step 1: Install Dependencies

In [ ]:
%%capture
# Official Unsloth install — auto-detects torch version & picks correct xformers
import os, re
if 'COLAB_' not in ''.join(os.environ.keys()):
    !pip install unsloth                          # Local / non-Colab setup
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
# Pin specific versions for stability
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
# Verify GPU is available
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

## 🤖 Step 2: Load Mistral 7B v0.3 with Unsloth (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

# ── CONFIG ──────────────────────────────────────────────
MAX_SEQ_LENGTH = 2048   # SQL queries are usually short; 2048 is sufficient
DTYPE          = None   # Auto-detect: float16 for T4, bfloat16 for A100
LOAD_IN_4BIT   = True   # QLoRA 4-bit quantization → fits on 16GB T4
# ────────────────────────────────────────────────────────

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = "unsloth/mistral-7b-v0.3-bnb-4bit",
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = DTYPE,
    load_in_4bit    = LOAD_IN_4BIT,
)

print("✅ Model loaded successfully!")

## ⚙️ Step 3: Apply QLoRA (PEFT Adapters)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                   = 16,      # LoRA rank — 16 is a good balance of quality vs speed
    target_modules      = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha          = 16,
    lora_dropout        = 0,       # 0 is optimized for Unsloth
    bias                = "none",  # "none" is optimized for Unsloth
    use_gradient_checkpointing = "unsloth",  # 30% less VRAM
    random_state        = 42,
    use_rslora          = False,
    loftq_config        = None,
)

# Print trainable parameter count
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")

## 📊 Step 4: Load & Prepare the SQL Dataset

We use `b-mc2/sql-create-context` — contains:
- `question`: Natural language question (e.g. "How many students passed?")
- `context`: CREATE TABLE schema (database structure)
- `answer`: The correct SQL query

This is the exact format used by SQLCoder and defog.ai.

In [ ]:
from datasets import load_dataset

# Load the SQLCoder dataset
dataset = load_dataset("b-mc2/sql-create-context", split="train")

print(f"✅ Dataset loaded: {len(dataset):,} examples")
print(f"\nColumns: {dataset.column_names}")
print(f"\n--- Sample Example ---")
sample = dataset[0]
print(f"Question : {sample['question']}")
print(f"Context  : {sample['context'][:200]}...")
print(f"Answer   : {sample['answer']}")

In [ ]:
# ── PROMPT TEMPLATE ────────────────────────────────────────────────────────
# Schema-aware SQL prompt — same style used by SQLCoder / defog.ai
# This teaches the model to use the table schema when generating SQL.
# ───────────────────────────────────────────────────────────────────────────

ALPACA_SQL_PROMPT = """Below is an instruction that describes a SQL task, paired with context that provides database schema information. Write a SQL query that correctly answers the request.

### Instruction:
{instruction}

### Context (Database Schema):
{context}

### SQL Query:
{response}"""

EOS_TOKEN = tokenizer.eos_token

def format_sql_prompt(examples):
    """Format each dataset row into the Alpaca SQL prompt template."""
    instructions = examples["question"]
    contexts     = examples["context"]
    responses    = examples["answer"]

    texts = []
    for instruction, context, response in zip(instructions, contexts, responses):
        text = ALPACA_SQL_PROMPT.format(
            instruction = instruction,
            context     = context,
            response    = response,
        ) + EOS_TOKEN
        texts.append(text)

    return {"text": texts}

# Apply formatting to full dataset
dataset = dataset.map(format_sql_prompt, batched=True)

print(f"✅ Dataset formatted!")
print(f"\n--- Formatted Example ---")
print(dataset[0]["text"][:600] + "...")

In [ ]:
# Check token length distribution to make sure MAX_SEQ_LENGTH is right
import matplotlib.pyplot as plt
import numpy as np

sample_texts = dataset.select(range(1000))["text"]
lengths = [len(tokenizer.encode(t)) for t in sample_texts]

print(f"Token length stats (sample of 1000):")
print(f"  Mean   : {np.mean(lengths):.0f}")
print(f"  Median : {np.median(lengths):.0f}")
print(f"  Max    : {np.max(lengths)}")
print(f"  95th % : {np.percentile(lengths, 95):.0f}")

plt.figure(figsize=(10, 4))
plt.hist(lengths, bins=50, color='steelblue', edgecolor='white')
plt.axvline(MAX_SEQ_LENGTH, color='red', linestyle='--', label=f'Max seq len ({MAX_SEQ_LENGTH})')
plt.xlabel("Token Length")
plt.ylabel("Count")
plt.title("SQL Prompt Token Length Distribution")
plt.legend()
plt.tight_layout()
plt.show()

## 🏋️ Step 5: Train the Model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model            = model,
    tokenizer        = tokenizer,
    train_dataset    = dataset,
    dataset_text_field = "text",
    max_seq_length   = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing          = False,   # Set True for short sequences to pack multiple into one

    args = TrainingArguments(
        # ── Batch & Gradient ─────────────────────────────
        per_device_train_batch_size    = 2,
        gradient_accumulation_steps    = 4,   # Effective batch size = 2×4 = 8

        # ── Epochs & Steps ───────────────────────────────
        num_train_epochs               = 1,   # Increase to 2-3 for better results
        # max_steps                    = 120, # Uncomment to do a quick test run

        # ── Learning Rate ────────────────────────────────
        warmup_steps                   = 5,
        learning_rate                  = 2e-4,
        lr_scheduler_type              = "linear",

        # ── Precision & Memory ───────────────────────────
        fp16                           = not torch.cuda.is_bf16_supported(),
        bf16                           = torch.cuda.is_bf16_supported(),
        optim                          = "adamw_8bit",

        # ── Logging & Saving ─────────────────────────────
        logging_steps                  = 10,
        output_dir                     = "queryforge-outputs",
        save_strategy                  = "epoch",

        # ── Reproducibility ──────────────────────────────
        seed                           = 42,
    ),
)

print("✅ Trainer configured!")

In [ ]:
# Show GPU memory before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"Max VRAM: {max_memory} GB")
print(f"Reserved before training: {start_gpu_memory} GB")
print("\n🚀 Starting training...")

In [ ]:
# 🚀 TRAIN!
trainer_stats = trainer.train()

# Training summary
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\n✅ Training complete!")
print(f"   Time elapsed   : {trainer_stats.metrics['train_runtime']:.0f}s ({trainer_stats.metrics['train_runtime']/60:.1f} min)")
print(f"   Samples/second : {trainer_stats.metrics['train_samples_per_second']:.2f}")
print(f"   Peak VRAM used : {used_memory} GB / {max_memory} GB")

## 🧪 Step 6: Inference — Test the Fine-tuned Model

In [ ]:
# Enable fast inference mode
FastLanguageModel.for_inference(model)

def generate_sql(question: str, schema: str, max_new_tokens: int = 256) -> str:
    """Generate SQL query from natural language question + schema."""
    prompt = ALPACA_SQL_PROMPT.format(
        instruction = question,
        context     = schema,
        response    = "",       # Leave blank — model will fill this in
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens = max_new_tokens,
        use_cache      = True,
        temperature    = 0.1,   # Low temp = deterministic SQL (good for accuracy)
        do_sample      = True,
    )

    # Decode only the newly generated tokens
    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return generated.strip()

print("✅ Inference mode enabled!")

In [ ]:
# ── TEST 1: E-commerce Query ─────────────────────────────────────────────
question = "Show the top 5 customers who spent the most in the last month"
schema   = """
CREATE TABLE customers (
    customer_id   INT PRIMARY KEY,
    customer_name VARCHAR(100),
    email         VARCHAR(100)
);
CREATE TABLE orders (
    order_id    INT PRIMARY KEY,
    customer_id INT,
    amount      DECIMAL(10,2),
    order_date  DATE,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);
"""

sql = generate_sql(question, schema)
print(f"Question : {question}")
print(f"\nGenerated SQL:")
print(sql)

In [ ]:
# ── TEST 2: HR / Employee Query ──────────────────────────────────────────
question = "List all employees in the Engineering department with salary above 80000"
schema   = """
CREATE TABLE departments (
    dept_id   INT PRIMARY KEY,
    dept_name VARCHAR(50)
);
CREATE TABLE employees (
    emp_id    INT PRIMARY KEY,
    emp_name  VARCHAR(100),
    salary    DECIMAL(10,2),
    dept_id   INT,
    FOREIGN KEY (dept_id) REFERENCES departments(dept_id)
);
"""

sql = generate_sql(question, schema)
print(f"Question : {question}")
print(f"\nGenerated SQL:")
print(sql)

In [ ]:
# ── TEST 3: Analytics / Aggregation Query ────────────────────────────────
question = "What is the total revenue per product category for Q3 2024?"
schema   = """
CREATE TABLE products (
    product_id   INT PRIMARY KEY,
    product_name VARCHAR(100),
    category     VARCHAR(50),
    price        DECIMAL(10,2)
);
CREATE TABLE sales (
    sale_id    INT PRIMARY KEY,
    product_id INT,
    quantity   INT,
    sale_date  DATE,
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
"""

sql = generate_sql(question, schema)
print(f"Question : {question}")
print(f"\nGenerated SQL:")
print(sql)

## 💾 Step 7: Save Model (Multiple Formats)

In [ ]:
# Save LoRA adapter weights locally
model.save_pretrained("queryforge-lora-adapter")
tokenizer.save_pretrained("queryforge-lora-adapter")
print("✅ LoRA adapter saved to: queryforge-lora-adapter/")

In [ ]:
# Save merged 16-bit model (for Hugging Face Hub)
model.save_pretrained_merged(
    "queryforge-merged-16bit",
    tokenizer,
    save_method = "merged_16bit",
)
print("✅ Merged 16-bit model saved to: queryforge-merged-16bit/")

In [ ]:
# Save as GGUF (for local inference with Ollama / LM Studio)
model.save_pretrained_gguf(
    "queryforge-gguf",
    tokenizer,
    quantization_method = "q4_k_m",  # 4-bit quantization — best quality/size tradeoff
)
print("✅ GGUF model saved to: queryforge-gguf/")
print("   → Import into Ollama: ollama import queryforge-gguf/model.gguf")

## 🚀 Step 8: Push to Hugging Face Hub (Optional)

In [ ]:
# Login to HuggingFace (get token from: https://huggingface.co/settings/tokens)
from huggingface_hub import login

HF_TOKEN    = "hf_YOUR_TOKEN_HERE"   # 🔑 Replace with your token
HF_USERNAME = "your-username"         # 👤 Replace with your HF username

login(token=HF_TOKEN)

REPO_NAME = f"{HF_USERNAME}/QueryForge-Mistral-7B-SQL"
print(f"Will push to: https://huggingface.co/{REPO_NAME}")

In [ ]:
# Push merged model to Hub
model.push_to_hub_merged(
    REPO_NAME,
    tokenizer,
    save_method    = "merged_16bit",
    token          = HF_TOKEN,
)
print(f"✅ Model pushed to: https://huggingface.co/{REPO_NAME}")

In [ ]:
# Push GGUF to Hub
model.push_to_hub_gguf(
    REPO_NAME + "-GGUF",
    tokenizer,
    quantization_method = "q4_k_m",
    token               = HF_TOKEN,
)
print(f"✅ GGUF pushed to: https://huggingface.co/{REPO_NAME}-GGUF")

## ✅ Done! Summary

| Step | Status |
|------|--------|
| Load Mistral 7B v0.3 (4-bit) | ✅ |
| Apply QLoRA adapters | ✅ |
| Load `b-mc2/sql-create-context` dataset | ✅ |
| Format as Alpaca SQL prompt | ✅ |
| Fine-tune with SFTTrainer | ✅ |
| Test inference | ✅ |
| Save LoRA / 16-bit / GGUF | ✅ |
| Push to Hugging Face Hub | ✅ |

---

### 🔗 Next Steps
- **Web App**: Check `app/` in the repo for a FastAPI + React frontend
- **Ollama**: `ollama import queryforge-gguf/model.gguf`
- **Improve**: Try `num_train_epochs=3` or `r=32` for better accuracy

**GitHub:** [QueryForge-AI](https://github.com/YOUR_USERNAME/QueryForge-AI)